In [ ]:
### building first, simple CNN baseline for protein stability prediction based on wt & mut seq ###

In [1]:
### 1. imports


# basic functionality
import os 
import numpy as np
import pandas as pd
import scipy
import sklearn.metrics as skmetrics

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import lightning as L

import torchmetrics
from torchmetrics.regression import PearsonCorrCoef

In [3]:
### running on gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
### 2. Dataloading ###

aa_alphabet = 'ACDEFGHIKLMNPQRSTVWY' # amino acid alphabet
aa_to_int = {aa: i for i, aa in enumerate(aa_alphabet)} # mapping from amino acid to number

# function to one hot encode sequence
def one_hot_encode(sequence):
    # initialize a zero matrix of shape (len(sequence), len(amino_acids))
    one_hot = torch.zeros(len(sequence), len(aa_alphabet))
    for i, aa in enumerate(sequence):
        # set the column corresponding to the amino acid to 1
        one_hot[i].scatter_(0, torch.tensor([aa_to_int[aa]]), 1)
    return one_hot



# sequence data, comes already batched, so treat accordingly in dataloader (batch_size=1)
class SequenceData(Dataset):
    def __init__(self, csv_file, label_col="ddG_ML"):
        """
        Initializes the dataset. 
        input:
            csv_file: path to the relevant data file, eg. "/home/data/mega_train.csv"
        """
        self.df = pd.read_csv(csv_file, sep=",")
        self.label_col = label_col
        # only have mutation rows
        self.df = self.df[self.df.mut_type!="wt"]
        # process the mutation row
        self.df["mutation_pos"] = self.df["mut_type"].apply(lambda x: int(x[1:-1])-1) # make position start at zero
        self.df["mutation_to"] = self.df["mut_type"].apply(lambda x: aa_to_int[x[-1]]) # give numerical label to mutation

        # group by wild type
        self.df = self.df.groupby("WT_name").agg(list)
        # get wild type names
        self.wt_names = self.df.index.values
        # precompute one-hot encoding for faster training
        self.encoded_seqs = {}
        for wt_name in self.wt_names:
            # get the correct row
            mut_row = self.df.loc[wt_name]
            seq = mut_row["wt_seq"][0]
            self.encoded_seqs[wt_name] = one_hot_encode(seq)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get the wild type name
        wt_name = self.wt_names[idx]
        # get the correct row
        mut_row = self.df.loc[wt_name]
        # get the wt sequence in one hot encoding
        sequence_encoding = self.encoded_seqs[wt_name]
            # dimensions (len, 20)

        # create mask and target tensors
        mask = torch.zeros((1, len(sequence_encoding),20)) # will be 1 where we have a measurement 
        target = torch.zeros((1, len(sequence_encoding),20)) # ddg values
            # remember batch size = 1
        # all mutations from df
        positions = torch.tensor(mut_row["mutation_pos"])
        amino_acids = torch.tensor(mut_row["mutation_to"])
        # get the labels
        labels = torch.tensor(mut_row[self.label_col])

        for i in range(len(sequence_encoding)):
            mask[0,i,amino_acids[positions==i]] = 1 # one where we have data (i.e. ddG measurement)
            target[0,i,amino_acids[positions==i]] = labels[positions==i] # fill with ddG values
        
        # returns encoded sequence, mask and target sequence 
        return {"sequence": sequence_encoding[None,:,:].float(), "mask": mask, "labels": target}


        # use this structure to
        # - predict ddG for all possible mutations
        # - mask serves to only compute loss at positions where experimental data exists

In [7]:
# usage
dataset_train = SequenceData('mega_train.csv')
dataset_val= SequenceData('mega_val.csv')
# dataset_test = SequenceData('project_data/mega_test.csv') # only once available

# use batch_size=1 bc we treat each sequence as one batch
dataloader_val = DataLoader(dataset_val, batch_size=1, shuffle=False)
dataloader_train = DataLoader(dataset_train, batch_size=1, shuffle=True)
# dataloader_test = DataLoader(dataset_test, batch_size=1, shuffle=False)

In [21]:
### 3. Data exploration ###

print('training data')
print(type(dataset_train))
# custom Pytorch Dataset object

print(len(dataset_train))
# nb of proteins

exemplary_protein = dataset_train[0]
print('exemplary datapoint (wt protein 0), which will be fed into CNN as one input:')
print(type(exemplary_protein))

print({k: v.shape for k,v in exemplary_protein.items()})
    # IMPORTANT: only 'sequence' will be used for model training, 'mask' and 'labels' both serve loss computation


print('\nvalidation data')
print(type(dataset_val))
# custom Pytorch Dataset object

print(len(dataset_val))
# nb of proteins

exemplary_protein = dataset_val[0]
print('exemplary datapoint (wt protein 0), which will be fed into CNN as one input:')
print(type(exemplary_protein))

print({k: v.shape for k,v in exemplary_protein.items()})
    # IMPORTANT: only 'sequence' will be used for model training, 'mask' and 'labels' both serve loss computation

training data
<class '__main__.SequenceData'>
239
exemplary datapoint (wt protein 0), which will be fed into CNN as one input:
<class 'dict'>
{'sequence': torch.Size([1, 63, 20]), 'mask': torch.Size([1, 63, 20]), 'labels': torch.Size([1, 63, 20])}

validation data
<class '__main__.SequenceData'>
31
exemplary datapoint (wt protein 0), which will be fed into CNN as one input:
<class 'dict'>
{'sequence': torch.Size([1, 39, 20]), 'mask': torch.Size([1, 39, 20]), 'labels': torch.Size([1, 39, 20])}


In [23]:
### 4. define CNN model architecture ###

# create new model class that inherits all properties from the torch.nn.Module
class simplecnn(nn.Module):
    # initialize
    def __init__(self):
        # load parent module
        super(simplecnn, self).__init__()

        # ingredients
        self.downconv = nn.Conv1d(in_channels=20, out_channels=64,kernel_size=3,padding=1)
        self.upconv = nn.Conv1d(in_channels=64, out_channels=20,kernel_size=1)

        # recipe
    def forward(self, original_input):
        permuteddim_input = original_input.permute(0,2,1) # permute dimensions from 1x20xlen(seq) to 20x1x63
        featuremaps1 = F.relu(self.downconv(permuteddim_input))
        output = self.upconv(featuremaps1)

        return output
        

In [ ]:
### 5. train model with lightning ###
class lightning_simplecnn(L.LightningModule):
    def __init__(self, learning_rate=1e-3):
        super().__init__()
